# <b> PART 2 </b>
### Phân Rã Ma Trận và Chéo Hóa

### <b> Helper Function </b>
Chứa các hàm tính toán ma trận cơ bản và thuật toán Jacobi để tìm giá trị riêng,vector riêng cho ma trận đối xứng.

In [1]:
import math

def transpose(matrix):
    """Tính ma trận chuyển vị."""
    return [[matrix[j][i] for j in range(len(matrix))] for i in range(len(matrix[0]))]

def matrix_multiply(A, B):
    """Nhân hai ma trận A và B."""
    m = len(A)
    n = len(A[0])
    p = len(B[0])
    result = [[0.0 for _ in range(p)] for _ in range(m)]
    for i in range(m):
        for j in range(p):
            for k in range(n):
                result[i][j] += A[i][k] * B[k][j]
    return result

def jacobi_eigenvalue(A, tol=1e-9, max_iter=100):
    """
    Thuật toán Jacobi tìm giá trị riêng và vector riêng cho ma trận đối xứng.
    """
    n = len(A)
    V = [[1.0 if i == j else 0.0 for j in range(n)] for i in range(n)]
    A_copy = [[A[i][j] for j in range(n)] for i in range(n)]

    for _ in range(max_iter):
        max_val = 0.0
        p, q = 0, 0
        for i in range(n):
            for j in range(i + 1, n):
                if abs(A_copy[i][j]) > max_val:
                    max_val = abs(A_copy[i][j])
                    p, q = i, j

        if max_val < tol:
            break

        # Xử lý trường hợp đặc biệt chia cho 0
        if A_copy[p][p] == A_copy[q][q]:
            theta = math.pi / 4 if A_copy[p][q] > 0 else -math.pi / 4
        else:
            theta = 0.5 * math.atan2(2 * A_copy[p][q], A_copy[p][p] - A_copy[q][q])

        c = math.cos(theta)
        s = math.sin(theta)

        A_pp = A_copy[p][p]
        A_qq = A_copy[q][q]
        A_pq = A_copy[p][q]

        A_copy[p][p] = c**2 * A_pp + 2*s*c*A_pq + s**2 * A_qq
        A_copy[q][q] = s**2 * A_pp - 2*s*c*A_pq + c**2 * A_qq
        A_copy[p][q] = A_copy[q][p] = 0.0

        for i in range(n):
            if i != p and i != q:
                A_ip = A_copy[i][p]
                A_iq = A_copy[i][q]
                A_copy[i][p] = A_copy[p][i] = c * A_ip + s * A_iq
                A_copy[i][q] = A_copy[q][i] = -s * A_ip + c * A_iq

            v_ip = V[i][p]
            v_iq = V[i][q]
            V[i][p] = c * v_ip + s * v_iq
            V[i][q] = -s * v_ip + c * v_iq

    eigenvalues = [A_copy[i][i] for i in range(n)]
    return eigenvalues, V

### <b> Hàm phân rã SVD </b>
Phân rã ma trận A thành U, Sigma, V^T

In [23]:
import math

def svd_decompose(A):
    m = len(A)
    n = len(A[0])

    A_T = transpose(A)
    M = matrix_multiply(A_T, A)

    eigenvalues, V = jacobi_eigenvalue(M, 1e-15, 500)

    eig_pairs = []
    for j in range(n):
        eig_val = eigenvalues[j] if eigenvalues[j] > 1e-10 else 0.0
        col = [V[i][j] for i in range(n)]
        eig_pairs.append((eig_val, col))

    eig_pairs.sort(key=lambda x: x[0], reverse=True)

    Sigma = [[0.0 for _ in range(n)] for _ in range(m)]
    V_sorted = [[0.0 for _ in range(n)] for _ in range(n)]
    singular_values = []

    for j in range(n):
        val = math.sqrt(eig_pairs[j][0])
        singular_values.append(val)
        if j < min(m, n):
            Sigma[j][j] = val
        for i in range(n):
            V_sorted[i][j] = eig_pairs[j][1][i]

    V_T = transpose(V_sorted)

    U = [[0.0 for _ in range(m)] for _ in range(m)]
    for j in range(min(m, n)):
        sigma = singular_values[j]
        if sigma > 1e-10:
            v_j = [V_sorted[i][j] for i in range(n)]
            Av = [sum(A[r][c] * v_j[c] for c in range(n)) for r in range(m)]
            for r in range(m):
                U[r][j] = Av[r] / sigma

            for k in range(j):
                dot = sum(U[r][k] * U[r][j] for r in range(m))
                for r in range(m):
                    U[r][j] -= dot * U[r][k]

            norm = math.sqrt(sum(U[r][j]**2 for r in range(m)))
            if norm > 1e-10:
                for r in range(m):
                    U[r][j] /= norm

    for j in range(m):
        col_norm = sum(U[r][j]**2 for r in range(m))
        if col_norm < 1e-10:
            for i in range(m):
                v = [1.0 if x == i else 0.0 for x in range(m)]
                for k in range(j):
                    dot = sum(U[r][k] * v[r] for r in range(m))
                    for r in range(m):
                        v[r] -= dot * U[r][k]
                norm = math.sqrt(sum(x**2 for x in v))
                if norm > 1e-10:
                    for r in range(m):
                        U[r][j] = v[r] / norm
                    break

    return U, Sigma, V_T

### <b> Hàm Chéo hóa ma trận (Diagonalization) </b>
Chéo hóa ma trận đối xứng A, trả về P, D, P_inv

In [16]:
def diagonalize_symmetric(A):
    """
    Chéo hóa ma trận đối xứng A.
    Trả về ma trận P, D, và P_inv sao cho A = P * D * P_inv.
    """
    n = len(A)

    # 1. Tìm giá trị riêng và vector riêng
    eigenvalues, P = jacobi_eigenvalue(A)

    D = [[0.0 for _ in range(n)] for _ in range(n)]
    for i in range(n):
        D[i][i] = eigenvalues[i]

    # 2. Tìm P^(-1)
    # Vì A đối xứng, P là ma trận trực giao, nên P^(-1) = P^T
    P_inv = transpose(P)

    return P, D, P_inv

### Numpy Test
Sử dụng các hàm có sẵn của <b> Numpy </b> để kiểm tra tính đúng đắn của code thuật toán SVD và Chéo hóa.

In [24]:
import numpy as np

def verify_svd(SVDTestCase: list):
    epsilon = 1e-10
    for i in range(len(SVDTestCase)):
        A = SVDTestCase[i]
        try:
            # Lấy kết quả từ hàm tự code
            U_f, Sigma_f, VT_f = svd_decompose(A)

            # Ép kiểu sang Numpy
            A_np = np.array(A, dtype=float)
            U_np = np.array(U_f, dtype=float)
            Sigma_np = np.array(Sigma_f, dtype=float)
            VT_np = np.array(VT_f, dtype=float)

            # Kiểm chứng tính toán học: A = U * Sigma * V^T
            A_reconstructed = U_np @ Sigma_np @ VT_np
            is_reconstructed = np.allclose(A_np, A_reconstructed, atol=epsilon)

            # Kiểm chứng trực chuẩn
            I_U = np.eye(U_np.shape[1])
            is_U_orthogonal = np.allclose(U_np.T @ U_np, I_U, atol=epsilon)

            I_V = np.eye(VT_np.shape[0])
            is_V_orthogonal = np.allclose(VT_np @ VT_np.T, I_V, atol=epsilon)

            # In kết quả
            if is_reconstructed and is_U_orthogonal and is_V_orthogonal:
                print(f"SVD: Đúng ở Test {i}")
            else:
                errors = []
                if not is_reconstructed: errors.append("A != U*Sigma*V^T")
                if not is_U_orthogonal:
                  errors.append("U không trực chuẩn")
                if not is_V_orthogonal: errors.append("V^T không trực chuẩn")
                print(f"SVD: Sai ở Test {i} ({', '.join(errors)})")

        except Exception as e:
            print(f"SVD: Sai ở Test {i} (Lỗi runtime: {e})")

def verify_diagonalize(DiagonalTestCase: list):
    epsilon = 1e-10
    for i in range(len(DiagonalTestCase)):
        A = DiagonalTestCase[i]
        try:
            # Lấy kết quả từ hàm tự code
            P_f, D_f, P_inv_f = diagonalize_symmetric(A)

            # Ép kiểu sang Numpy
            A_np = np.array(A, dtype=float)
            P_np = np.array(P_f, dtype=float)
            D_np = np.array(D_f, dtype=float)
            P_inv_np = np.array(P_inv_f, dtype=float)

            # Kiểm chứng tái tạo: A = P * D * P_inv
            A_reconstructed = P_np @ D_np @ P_inv_np
            is_reconstructed = np.allclose(A_np, A_reconstructed, atol=epsilon)

            # Kiểm chứng nghịch đảo: P * P_inv = I
            I_P = np.eye(P_np.shape[0])
            is_inverse = np.allclose(P_np @ P_inv_np, I_P, atol=epsilon)

            if is_reconstructed and is_inverse:
                print(f"Diagonalize: Đúng ở Test {i}")
            else:
                errors = []
                if not is_reconstructed: errors.append("A != P*D*P^-1")
                if not is_inverse: errors.append("P_inv không phải nghịch đảo của P")
                print(f"Diagonalize: Sai ở Test {i} ({', '.join(errors)})")

        except Exception as e:
            print(f"Diagonalize: Sai ở Test {i} (Lỗi runtime: {e})")

def verify_part2(SVDTestCase: list, DiagonalTestCase: list):
    print("\n--- TEST SVD ---")
    verify_svd(SVDTestCase)
# ==========================================
    print("\n--- TEST DIAGONALIZATION ---")
    verify_diagonalize(DiagonalTestCase)

<b> BỘ TESTCASE MẪU PART 2 </b>

In [26]:
# 1. Testcase cho Phân rã SVD:
SVDTestCase = [
    # Test 0: Ma trận vuông 3x3 (Cơ bản)
    [
        [3, 1, 1],
        [-1, 3, 1],
        [1, -1, 3]
    ],

    # Test 1: Ma trận 3x2 (m > n)
    [
        [1, 2],
        [3, 4],
        [5, 6]
    ],

    # Test 2: Ma trận 2x3 (m < n)
    [
        [1, 3, 5],
        [2, 4, 6]
    ],

    # Test 3: Ma trận trực giao / quay 2x2
    [
        [0, -1],
        [1, 0]
    ],

    # Test 4: Ma trận toàn số 0 (Góc cạnh)
    [
        [0, 0],
        [0, 0]
    ],

    # Test 5: Ma trận đường chéo chứa giá trị âm
    [
        [3, 0, 0],
        [0, -2, 0],
        [0, 0, 5]
    ],

    # Test 6: Ma trận suy biến / hạng không đầy đủ (hàng 2 = 2 * hàng 1)
    [
        [1, 2],
        [2, 4]
    ],

    # Test 7: Ma trận vuông 4x4 lớn hơn
    [
        [1, 2, 3, 4],
        [5, 6, 7, 8],
        [9, 10, 11, 12],
        [13, 14, 15, 16]
    ],

    # Test 8: Ma trận cột (3x1)
    [
        [1],
        [2],
        [3]
    ],

    # Test 9: Ma trận dòng (1x3)
    [
        [1, 2, 3]
    ]
]

# 2. Testcase cho Chéo hóa: (Sử dụng ma trận đối xứng)
DiagonalTestCase = [
    [
        [4, 1],
        [1, 3]
    ],
    # Test 1: Ma trận đối xứng 3x3
    [
        [2, -1, 0],
        [-1, 2, -1],
        [0, -1, 2]
    ],
    # Test 2: Ma trận đã là đường chéo
    [
        [5, 0],
        [0, -3]
    ],

    [
        [0, -1],
        [1, 0]
    ]
]

# GỌI HÀM KIỂM CHỨNG PART 2
verify_part2(SVDTestCase, DiagonalTestCase)


--- TEST SVD ---
SVD: Đúng ở Test 0
SVD: Đúng ở Test 1
SVD: Đúng ở Test 2
SVD: Đúng ở Test 3
SVD: Đúng ở Test 4
SVD: Đúng ở Test 5
SVD: Đúng ở Test 6
SVD: Đúng ở Test 7
SVD: Đúng ở Test 8
SVD: Đúng ở Test 9

--- TEST DIAGONALIZATION ---
Diagonalize: Đúng ở Test 0
Diagonalize: Đúng ở Test 1
Diagonalize: Đúng ở Test 2
Diagonalize: Sai ở Test 3 (A != P*D*P^-1)
